First Pipeline to run binding energy


In [1]:
!uv pip install fairchem-core

Using Python 3.13.11 environment at: C:\Users\miaom\miniconda3\envs\cms
Audited 1 package in 4.98s


In [2]:
# check through Hugging Face Hub if the package is installed correctly, paste result in chatgpt or claude to confirm
# you mainly need to check off " Read access to contents of all public gated repos you can access" and " Read access to contents of all private gated repos you can access" in tokens settings in Hugging Face Hub, and that the token is correctly set up in your environment variables
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '69d423a991dfba6b651bddfe', 'name': 'umami393', 'fullname': 'Margaret', 'isPro': False, 'avatarUrl': '/avatars/eb54fe80196fee8cdf7c5c7ace19b460.svg', 'orgs': [{'type': 'org', 'id': '64374111a701a7e744c02b0e', 'name': 'princetonu', 'fullname': 'Princeton University', 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/68e396f2b5bb631e9b2fac9a/b3xXusq8Zz3ej8Z6fRTSZ.png'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Rep Perm', 'role': 'fineGrained', 'createdAt': '2026-04-06T21:24:55.150Z', 'fineGrained': {'canReadGatedRepos': False, 'global': [], 'scoped': [{'entity': {'_id': '67098f31ee3ea12411d3f44e', 'type': 'model', 'name': 'facebook/OMAT24'}, 'permissions': []}, {'entity': {'_id': '67f5bb06b8e3712b3da49bfc', 'type': 'model', 'name': 'facebook/UMA'}, 'permissions': []}, {'entity': {'_id': '69d423a991dfba6b651bddfe', 'type': 'user', 'name': 'umami393'}, 'permissions': ['repo.content.read']}]}}}}


In [28]:
from fairchem.core import pretrained_mlip

predictor = pretrained_mlip.get_predict_unit(
    model_name="uma-s-1p1",
    device="cpu"
)

In [29]:
# test with calculator 
from ase.build import bulk
from fairchem.core import pretrained_mlip, FAIRChemCalculator

device = "cpu"

atoms = bulk("Si")

model_name = "uma-s-1p1"
predictor = pretrained_mlip.get_predict_unit(model_name)
calc = FAIRChemCalculator(predictor, task_name="omat")

atoms.calc = calc
e = atoms.get_potential_energy()
print(e)



-10.82311692799609


In [30]:
from ase.io import read
from ase.build import molecule
from ase.optimize import BFGS

# -------------------------
# Load structures
# -------------------------
mof_74 = read("mg_mof74.cif")
co2 = molecule("CO2")

# -------------------------
# Assign calculator
# -------------------------
mof_74.calc = calc
co2.calc = calc

# -------------------------
# Relax MOF
# -------------------------
opt_mof = BFGS(mof_74, logfile="opt_mof_74.log")
opt_mof.run(fmax=0.01)

E_mof = mof_74.get_potential_energy()
print(f"E(Mg-MOF-74) = {E_mof:.6f} eV")

# Relax CO2
co2.center(vacuum=10.0)  # IMPORTANT (no periodic interactions)

opt_co2 = BFGS(co2, logfile="opt_co2.log")
opt_co2.run(fmax=0.01)

E_co2 = co2.get_potential_energy()
print(f"E(CO2, free) = {E_co2:.6f} eV")

E(Mg-MOF-74) = -1177.415341 eV
E(CO2, free) = -22.596883 eV


In [40]:
import numpy as np
# get indices of metal atoms (Mg)
metal_indices = [i for i, a in enumerate(mof_74) if a.symbol == "Mg"]
metal_positions = mof_74.positions[metal_indices]

In [41]:
# center of unit cell (cartesian)
cell_center = np.dot([0.5, 0.5, 0.5], mof_74.get_cell())

def get_direction(metal_pos):
    direction = cell_center - metal_pos
    return direction / np.linalg.norm(direction)

In [42]:
from ase.geometry import get_distances
import numpy as np

def is_valid_position_full(mof, co2, cutoff=1.6):
    dmat = get_distances(
        mof.positions,
        co2.positions,
        cell=mof.get_cell(),
        pbc=mof.get_pbc()
    )[1]
    return np.min(dmat) > cutoff

In [43]:
def place_co2_oriented(mof, co2, metal_index):
    metal_pos = mof.positions[metal_index]

    cell_center = np.dot([0.5, 0.5, 0.5], mof.get_cell())
    direction = cell_center - metal_pos
    direction /= np.linalg.norm(direction)

    for d_mg_o in np.linspace(2.2, 2.8, 7):

        co2_copy = co2.copy()

        # align CO2 axis
        co2_axis = co2_copy.positions[2] - co2_copy.positions[0]
        co2_axis /= np.linalg.norm(co2_axis)

        v = np.cross(co2_axis, direction)
        angle = np.degrees(np.arccos(np.clip(np.dot(co2_axis, direction), -1, 1)))

        if np.linalg.norm(v) > 1e-8:
            co2_copy.rotate(angle, v, center='COM')

        # place oxygen
        target = metal_pos + d_mg_o * direction
        shift = target - co2_copy.positions[0]
        co2_copy.positions += shift

        # IMPORTANT: correct function call
        if is_valid_position_full(mof, co2_copy):
            return co2_copy

    return None

In [44]:
# combine system + computing binding energy 
def compute_binding_energy(mof, co2_placed):
    system = mof.copy()
    system += co2_placed

    system.calc = calc

    opt = BFGS(system, logfile=None)
    opt.run(fmax=0.05)

    E_total = system.get_potential_energy()

    return E_total - E_mof - E_co2

In [ ]:
# testing only for one magnesium site first (we can loop over all sites later)
mg_idx = 14
print(f"Testing Mg site index: {mg_idx}, symbol = {mof_74[mg_idx].symbol}")

if mof_74[mg_idx].symbol != "Mg":
    raise ValueError(f"Atom index {mg_idx} is not Mg. It is {mof_74[mg_idx].symbol}.")

co2_placed = place_co2_oriented(mof_74, co2, mg_idx)

if co2_placed is None:
    raise RuntimeError("Could not place CO2 near Mg without overlap.")

print("CO2 placed successfully")
print("CO2 positions:\n", co2_placed.positions)

Testing Mg site index: 14, symbol = Mg
CO2 placed successfully
CO2 positions:
 [[8.82231721 3.23474244 1.55310651]
 [9.1357771  2.12758286 1.2903883 ]
 [8.50885984 4.34189321 1.81582265]]


In [ ]:
from ase.io import read, write

# intialize combined system for inspection before relaxation
system = mof_74.copy()
system += co2_placed
system.calc = calc

# inspect Mg-O distances before relaxation
print("Initial Mg–O distances:")
for j in range(len(mof_74), len(system)):
    if system[j].symbol == "O":
        d = system.get_distance(mg_idx, j, mic=True)
        print(d)


# -------------------------
# Relax combined system
# -------------------------
opt_system = BFGS(system, logfile="opt_single_site.log")
opt_system.run(fmax=0.05)

E_total = system.get_potential_energy()
E_bind = E_total - E_mof - E_co2

print(f"E(total)   = {E_total:.6f} eV")
print(f"E(bind)    = {E_bind:.6f} eV")

# -------------------------
# Inspect Mg-O distances after relaxation
# -------------------------
co2_start = len(mof_74)
for j in range(co2_start, len(system)):
    if system[j].symbol == "O":
        d = system.get_distance(mg_idx, j, mic=True)
        print(f"Mg-O distance to O atom {j}: {d:.3f} Å")



Initial Mg–O distances:
1.6197117768940394
3.980278834062521
E(total)   = -1200.531405 eV
E(bind)    = -0.519181 eV
Mg-O distance to O atom 163: 2.364 Å
Mg-O distance to O atom 164: 4.504 Å


In [51]:
from ase.visualize import view

view(system)

<Popen: returncode: None args: ['c:\\Users\\miaom\\miniconda3\\envs\\cms\\py...>

In [ ]:
binding_energies = []

for idx in metal_indices:
    co2_placed = place_co2_oriented(mof_74, co2, idx)

    if co2_placed is None:
        print(f"Skipping site {idx}")
        continue

    system = mof_74.copy()
    system += co2_placed
    system.calc = calc

    # freeze MOF
    from ase.constraints import FixAtoms
    mask = [True]*len(mof_74) + [False]*len(co2_placed)
    system.set_constraint(FixAtoms(mask=mask))

    opt = BFGS(system, logfile=None)
    opt.run(fmax=0.05)

    E_total = system.get_potential_energy()
    E_bind = E_total - E_mof - E_co2

    binding_energies.append(E_bind)

    print(f"Site {idx}: {E_bind:.4f} eV")

Skipping site 0 (no valid placement)


KeyboardInterrupt: 